# MixNet V3 â€” True Cross-Mixture Training
**NMR Blind Source Separation with Cross-Mixture Attention**

Upload `NMR_PROJECT_FINAL_PACKAGE.zip` to your Google Drive before running.

In [ ]:
# ============================================================
# CELL 1: Mount Drive + Setup
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
# Check GPU
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - go to Runtime > Change runtime type > GPU')
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# ============================================================
# CELL 2: Extract Data
# Change ZIP_PATH to wherever you put the zip in Drive
# ============================================================
import zipfile, os

ZIP_PATH = '/content/drive/MyDrive/NMR_PROJECT_FINAL_PACKAGE.zip'  # <-- change if needed
EXTRACT_DIR = '/content/data'

if not os.path.exists(EXTRACT_DIR):
    print('Extracting...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print('Done!')
else:
    print('Already extracted!')

DATA_DIR = '/content/data'
print('Files:', os.listdir(DATA_DIR))

In [ ]:
# ============================================================
# CELL 3: Data Loader
# ============================================================
import glob, numpy as np, pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

class RealNMRDataset(Dataset):
    def __init__(self, data_dir, normalize=True, max_datasets=None):
        self.data_dir = data_dir
        self.normalize = normalize
        
        meta_path = os.path.join(data_dir, 'mixture_metadata.csv')
        self.metadata = pd.read_csv(meta_path)
        
        comp_dir = os.path.join(data_dir, 'components', 'npy')
        self.compound_spectra = {}
        for f in glob.glob(os.path.join(comp_dir, '*.npy')):
            name = os.path.basename(f).replace('.npy', '')
            spec = np.load(f).astype(np.float32)
            if normalize:
                spec = spec / spec.max()
            self.compound_spectra[name] = spec
        
        self.dataset_ids = sorted(self.metadata['dataset_id'].unique())
        if max_datasets:
            self.dataset_ids = self.dataset_ids[:max_datasets]
        
        self.dataset_meta = {}
        for ds_id in self.dataset_ids:
            self.dataset_meta[ds_id] = self.metadata[
                self.metadata['dataset_id'] == ds_id
            ].reset_index(drop=True)
    
    def __len__(self):
        return len(self.dataset_ids)
    
    def __getitem__(self, idx):
        ds_id = self.dataset_ids[idx]
        ds_meta = self.dataset_meta[ds_id]
        
        npz_path = os.path.join(self.data_dir, 'mixtures', f'{ds_id}.npz')
        npz_data = np.load(npz_path)
        mixtures = npz_data['spectra'].astype(np.float32)
        
        if self.normalize:
            mix_max = mixtures.max(axis=1, keepdims=True)
            mix_max = np.where(mix_max > 0, mix_max, 1.0)
            mixtures = mixtures / mix_max
        
        row0 = ds_meta.iloc[0]
        compound_names = [row0[f'compound_{k}'] for k in range(1, 6)]
        compounds = np.stack([self.compound_spectra[name] for name in compound_names], axis=0)
        
        concentrations = np.zeros((20, 5), dtype=np.float32)
        for i in range(len(ds_meta)):
            for j in range(5):
                concentrations[i, j] = ds_meta.iloc[i][f'fraction_{j+1}']
        
        return {
            'mixtures': torch.from_numpy(mixtures),
            'compounds': torch.from_numpy(compounds),
            'concentrations': torch.from_numpy(concentrations),
            'dataset_id': ds_id
        }

def create_loaders(data_dir, batch_size=1, seed=42, max_datasets=None):
    dataset = RealNMRDataset(data_dir, max_datasets=max_datasets)
    total = len(dataset)
    np.random.seed(seed)
    idx = np.random.permutation(total)
    n_train = int(total * 0.8)
    n_val = int(total * 0.1)
    train_set = torch.utils.data.Subset(dataset, idx[:n_train])
    val_set = torch.utils.data.Subset(dataset, idx[n_train:n_train+n_val])
    test_set = torch.utils.data.Subset(dataset, idx[n_train+n_val:])
    print(f'Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}')
    return (DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2),
            DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=2),
            DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2))

print('Data loader ready!')

In [ ]:
# ============================================================
# CELL 4: MixNet V3 Architecture
# ============================================================
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, **kwargs):
        super().__init__()
        pad = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size, padding=pad), nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True),
            nn.Conv1d(out_ch, out_ch, kernel_size, padding=pad), nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.block(x)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool_k=4, pool_s=4):
        super().__init__()
        self.conv = ConvBlock(in_ch, out_ch)
        self.pool = nn.MaxPool1d(pool_k, pool_s)
    def forward(self, x):
        s = self.conv(x)
        return s, self.pool(s)

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, scale=4):
        super().__init__()
        self.up = nn.ConvTranspose1d(in_ch, in_ch, kernel_size=scale, stride=scale)
        self.conv = ConvBlock(in_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        return self.conv(torch.cat([self.up(x), skip], dim=1))

class SharedEncoder(nn.Module):
    def __init__(self, ch=[64,128,256,256,512]):
        super().__init__()
        self.enc1 = DownBlock(1, ch[0])
        self.enc2 = DownBlock(ch[0], ch[1])
        self.enc3 = DownBlock(ch[1], ch[2])
        self.enc4 = DownBlock(ch[2], ch[3])
        self.enc5 = DownBlock(ch[3], ch[4])
        self.bn = ConvBlock(ch[4], ch[4])
    def forward(self, x):
        s1,x = self.enc1(x); s2,x = self.enc2(x); s3,x = self.enc3(x)
        s4,x = self.enc4(x); s5,x = self.enc5(x)
        return self.bn(x), [s1,s2,s3,s4,s5]

class CrossMixtureAttention(nn.Module):
    def __init__(self, d_model=512, nhead=8, num_layers=2):
        super().__init__()
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                    dim_feedforward=1024, dropout=0.1, batch_first=True)
        self.attn = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x):  # x: (B, M, C, L)
        B,M,C,L = x.shape
        x = x.permute(0,3,1,2).reshape(B*L, M, C)
        x = self.norm(self.attn(x))
        return x.reshape(B,L,M,C).permute(0,2,3,1)

class MixNetV3(nn.Module):
    def __init__(self, in_channels=20, out_channels=5, ch=[64,128,256,256,512]):
        super().__init__()
        self.M = in_channels
        self.encoder = SharedEncoder(ch)
        self.cross_attn = CrossMixtureAttention(d_model=ch[4], nhead=8, num_layers=2)
        self.aggregate = nn.Sequential(
            nn.Conv1d(ch[4]*in_channels, ch[4], 1), nn.ReLU(),
            nn.Conv1d(ch[4], ch[4], 1), nn.ReLU()
        )
        self.dec5 = UpBlock(ch[4], ch[4], ch[3])
        self.dec4 = UpBlock(ch[3], ch[3], ch[2])
        self.dec3 = UpBlock(ch[2], ch[2], ch[1])
        self.dec2 = UpBlock(ch[1], ch[1], ch[0])
        self.dec1 = UpBlock(ch[0], ch[0], ch[0])
        self.final = nn.Conv1d(ch[0], out_channels, 1)
        self.act = nn.Softplus()
    
    def forward(self, x):  # x: (B, M, L)
        B,M,L = x.shape
        bottlenecks, all_skips = [], [[] for _ in range(5)]
        for i in range(M):
            bn, skips = self.encoder(x[:,i:i+1,:])
            bottlenecks.append(bn)
            for j in range(5): all_skips[j].append(skips[j])
        encoded = torch.stack(bottlenecks, dim=1)  # (B,M,C,L)
        attended = self.cross_attn(encoded)
        agg = self.aggregate(attended.reshape(B, M*attended.shape[2], attended.shape[3]))
        avg_skips = [torch.stack(all_skips[j],dim=1).mean(dim=1) for j in range(5)]
        x = self.dec5(agg, avg_skips[4])
        x = self.dec4(x, avg_skips[3])
        x = self.dec3(x, avg_skips[2])
        x = self.dec2(x, avg_skips[1])
        x = self.dec1(x, avg_skips[0])
        return self.act(self.final(x))

# Test
model = MixNetV3()
params = sum(p.numel() for p in model.parameters())
print(f'MixNet V3 params: {params:,}')
with torch.no_grad():
    y = model(torch.randn(1,20,16384))
print(f'Output shape: {y.shape}')

In [ ]:
# ============================================================
# CELL 5: Training
# ============================================================
import time

EPOCHS = 60
PATIENCE = 15
LR = 1e-4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Training on:', device)

train_loader, val_loader, test_loader = create_loaders(DATA_DIR, batch_size=1)

model = MixNetV3(in_channels=20, out_channels=5).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
mse = nn.MSELoss()

best_val = float('inf')
patience_ctr = 0
os.makedirs('/content/drive/MyDrive/nmr_checkpoints', exist_ok=True)

for epoch in range(EPOCHS):
    # Train
    model.train()
    tl, ts, tr, tb = 0,0,0,0
    t0 = time.time()
    for batch in train_loader:
        m = batch['mixtures'].to(device)
        c = batch['compounds'].to(device)
        a = batch['concentrations'].to(device)
        if m.shape[0]==1: m,c,a = m.squeeze(0).unsqueeze(0),c.squeeze(0).unsqueeze(0),a.squeeze(0).unsqueeze(0)
        optimizer.zero_grad()
        pred = model(m)
        sl = mse(pred, c)
        rl = mse(torch.bmm(a, pred), m)
        loss = sl + 0.5*rl
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tl+=loss.item(); ts+=sl.item(); tr+=rl.item(); tb+=1
    
    # Validate
    model.eval()
    vl, vb = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            m = batch['mixtures'].to(device)
            c = batch['compounds'].to(device)
            a = batch['concentrations'].to(device)
            if m.shape[0]==1: m,c,a = m.squeeze(0).unsqueeze(0),c.squeeze(0).unsqueeze(0),a.squeeze(0).unsqueeze(0)
            pred = model(m)
            sl = mse(pred, c)
            rl = mse(torch.bmm(a, pred), m)
            vl += (sl + 0.5*rl).item(); vb+=1
    
    scheduler.step()
    val_loss = vl/vb if vb>0 else 0
    elapsed = time.time()-t0
    
    status = ''
    if val_loss < best_val:
        best_val = val_loss
        patience_ctr = 0
        torch.save({'epoch':epoch+1,'model_state_dict':model.state_dict(),'val_loss':best_val},
                   '/content/drive/MyDrive/nmr_checkpoints/best_model_v3.pth')
        status = '<< BEST (saved to Drive)'
    else:
        patience_ctr += 1
        status = f'(patience: {patience_ctr}/{PATIENCE})'
    
    print(f'Epoch {epoch+1:>3}/{EPOCHS} | Train: {tl/tb:.6f} (spec={ts/tb:.6f}) | Val: {val_loss:.6f} | {elapsed:.1f}s {status}')
    
    if patience_ctr >= PATIENCE:
        print(f'Early stopping at epoch {epoch+1}!')
        break

print(f'\nBest val loss: {best_val:.6f}')

In [ ]:
# ============================================================
# CELL 6: TRUE BLIND TEST (unseen compounds)
# ============================================================
from itertools import permutations

spectral_length = 16384
ppm = np.linspace(10.0, 0.0, spectral_length)

def make_peak(center, width=0.02, height=1.0):
    return height * np.exp(-0.5 * ((ppm - center) / width) ** 2)

# 5 NEW compounds â€” model has NEVER seen these
comps = {}
c_a = make_peak(0.85, height=1.0) + make_peak(1.55, height=0.6) + make_peak(5.30, height=0.3, width=0.03)
comps['A'] = c_a/c_a.max()
c_b = make_peak(2.75, height=1.0) + make_peak(6.85, height=0.8) + make_peak(7.45, height=0.5)
comps['B'] = c_b/c_b.max()
c_c = make_peak(1.95, height=0.7) + make_peak(3.85, height=1.0) + make_peak(8.20, height=0.4, width=0.04)
comps['C'] = c_c/c_c.max()
c_d = make_peak(0.95, height=1.0) + make_peak(4.55, height=0.5) + make_peak(6.10, height=0.7)
comps['D'] = c_d/c_d.max()
c_e = make_peak(3.15, height=1.0) + make_peak(5.70, height=0.6) + make_peak(9.20, height=0.3, width=0.05)
comps['E'] = c_e/c_e.max()

names = list(comps.keys())
true_m = np.stack([comps[n] for n in names])

np.random.seed(777)
conc = np.random.dirichlet([1,1,1,1,1], size=20).astype(np.float32)
mixtures = conc @ true_m
for i in range(20): mixtures[i] /= mixtures[i].max()

mix_t = torch.from_numpy(mixtures.astype(np.float32)).unsqueeze(0).to(device)

# Load best V3
ckpt = torch.load('/content/drive/MyDrive/nmr_checkpoints/best_model_v3.pth', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

with torch.no_grad():
    pred = model(mix_t).squeeze(0).cpu().numpy()

# Best assignment
best_corr, best_corrs, best_perm = -1, None, None
for p in permutations(range(5)):
    corrs = [float(np.corrcoef(true_m[i], pred[j])[0,1]) for i,j in enumerate(p)]
    avg = np.mean(corrs)
    if avg > best_corr: best_corr=avg; best_corrs=corrs; best_perm=p

print('='*60)
print('  TRUE BLIND TEST â€” V3 (Cross-Mixture Attention)')
print('='*60)
for i,j in enumerate(best_perm):
    print(f'  Compound {names[i]} -> Pred #{j+1}: {best_corrs[i]:.4f}')
print(f'  AVERAGE: {best_corr:.4f}')
print()
print('V1 baseline was: 0.5203')
diff = best_corr - 0.5203
print(f'Improvement: {diff:+.4f} ({diff/0.5203*100:+.1f}%)')